In [ ]:
import os
import json
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torchvision import transforms, models
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Set device and model directory
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_DIR = "/content/drive/MyDrive/Personal /autism_project"

print(f"✅ Using device: {device}")

Mounted at /content/drive
✅ Using device: cuda


In [ ]:
def load_emotion_pipeline(model_dir):
    # 1. Load emotion class ordering from class_mapping_expression.json
    expr_map_path = os.path.join(model_dir, "class_mapping_expression.json")
    with open(expr_map_path, "r") as f:
        expr_map = json.load(f)

    if isinstance(expr_map, list):
        emotion_classes = expr_map
    elif isinstance(expr_map, dict):
        first_val = next(iter(expr_map.values()))
        if isinstance(first_val, list):
            emotion_classes = first_val
        elif str(next(iter(expr_map.keys()))).isdigit():
            emotion_classes = [v for k, v in sorted(expr_map.items(), key=lambda item: int(item[0]))]
        else:
            emotion_classes = [k for k, v in sorted(expr_map.items(), key=lambda item: int(item[1]))]
    else:
        emotion_classes = ['Natural', 'anger', 'fear', 'joy', 'sadness', 'surprise']

    # 2. Preprocessing pipeline matching training
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.Grayscale(num_output_channels=3),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    # 3. Initialize ResNet-18 and load expression weights
    emotion_model = models.resnet18(weights=None)
    emotion_model.fc = nn.Linear(emotion_model.fc.in_features, len(emotion_classes))
    emotion_model.load_state_dict(
        torch.load(os.path.join(model_dir, "best_emotion_resnet18_final.pth"), map_location=device)
    )
    emotion_model = emotion_model.to(device).eval()

    return {
        "emotion_classes": emotion_classes,
        "transform": transform,
        "emotion_model": emotion_model
    }

emotion_pipeline = load_emotion_pipeline(MODEL_DIR)
print("✅ Loaded classes:", emotion_pipeline["emotion_classes"])
print("✅ Facial Expression Pipeline ready.")

✅ Loaded classes: ['Natural', 'anger', 'fear', 'joy', 'sadness', 'surprise']
✅ Facial Expression Pipeline ready.


In [ ]:
import matplotlib.pyplot as plt

def extract_face(image_path):
    """Detects and crops the largest face using OpenCV Haar Cascades with safe portrait fallback."""
    img_bgr = cv2.imread(image_path)
    if img_bgr is None:
        raise ValueError(f"Could not read image at {image_path}")

    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    h, w = gray.shape

    face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(60, 60))

    if len(faces) > 0:
        # Select largest detected face
        x, y, fw, fh = max(faces, key=lambda rect: rect[2] * rect[3])
        pad_top = int(0.30 * fh)
        pad_bottom = int(0.40 * fh)
        pad_side = int(0.25 * fw)

        x1 = max(0, x - pad_side)
        y1 = max(0, y - pad_top)
        x2 = min(w, x + fw + pad_side)
        y2 = min(h, y + fh + pad_bottom)
        cropped = img_rgb[y1:y2, x1:x2]
        detection_status = "Face Detected"
    else:
        # Lower-center portrait fallback (avoids ceiling/wall crops)
        crop_size = int(min(w, h) * 0.65)
        cx, cy = w // 2, int(h * 0.55)
        x1 = max(0, cx - crop_size // 2)
        y1 = max(0, cy - crop_size // 2)
        x2 = min(w, x1 + crop_size)
        y2 = min(h, y1 + crop_size)
        cropped = img_rgb[y1:y2, x1:x2]
        detection_status = "Center Portrait Fallback"

    return Image.fromarray(cropped), detection_status

def predict_facial_expression(image_path):
    """Runs facial expression classification and displays the cropped face + probability breakdown."""
    face_pil, status = extract_face(image_path)
    input_tensor = emotion_pipeline["transform"](face_pil).unsqueeze(0).to(device)
    classes = emotion_pipeline["emotion_classes"]

    with torch.no_grad():
        logits = emotion_pipeline["emotion_model"](input_tensor)
        probs = F.softmax(logits, dim=1).squeeze(0).cpu().numpy()

    # Sort predictions from highest to lowest confidence
    ranked = sorted(zip(classes, probs), key=lambda x: x[1], reverse=True)
    top_emotion, top_conf = ranked[0]

    # Display cropped face alongside probability bar chart
    fig, axes = plt.subplots(1, 2, figsize=(9, 4))
    axes[0].imshow(face_pil)
    axes[0].set_title(f"Cropped Input ({status})")
    axes[0].axis("off")

    labels = [r[0].capitalize() for r in ranked][::-1]
    scores = [r[1] * 100 for r in ranked][::-1]
    colors = ["#2563eb" if s == max(scores) else "#94a3b8" for s in scores]

    axes[1].barh(labels, scores, color=colors)
    axes[1].set_xlim(0, 100)
    axes[1].set_xlabel("Confidence (%)")
    axes[1].set_title(f"Predicted: {top_emotion.capitalize()} ({top_conf * 100:.1f}%)")
    plt.tight_layout()
    plt.show()

    # Print formatted report
    print("==========================================================")
    print("             FACIAL EXPRESSION ANALYSIS REPORT            ")
    print("==========================================================")
    print(f"- Dominant Expression : {top_emotion.capitalize()}")
    print(f"- Confidence Score    : {top_conf * 100:.2f}%")
    print("----------------------------------------------------------")
    print("Full Expression Breakdown:")
    for idx, (label, prob) in enumerate(ranked, 1):
        print(f"{idx}. {label.capitalize():<12} : {prob * 100:.2f}%")
    print("==========================================================")

In [ ]:
import ipywidgets as widgets
from IPython.display import display, Javascript, clear_output
from google.colab.output import eval_js
from google.colab import files
from base64 import b64decode

def capture_webcam_colab(filename="webcam_capture.jpg", quality=0.9):
    js = Javascript('''
        async function takePhoto(quality) {
            const div = document.createElement('div');
            const capture = document.createElement('button');
            capture.textContent = '📸 Click to Snap Photo';
            capture.style.padding = '10px 20px';
            capture.style.fontSize = '15px';
            capture.style.marginBottom = '10px';
            capture.style.backgroundColor = '#2563eb';
            capture.style.color = 'white';
            capture.style.border = 'none';
            capture.style.borderRadius = '6px';
            capture.style.cursor = 'pointer';
            div.appendChild(capture);

            const video = document.createElement('video');
            video.style.display = 'block';
            video.style.maxWidth = '500px';
            video.style.borderRadius = '8px';
            const stream = await navigator.mediaDevices.getUserMedia({video: true});

            document.body.appendChild(div);
            div.appendChild(video);
            video.srcObject = stream;
            await video.play();

            google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);

            await new Promise((resolve) => capture.onclick = resolve);

            const canvas = document.createElement('canvas');
            canvas.width = video.videoWidth;
            canvas.height = video.videoHeight;
            canvas.getContext('2d').drawImage(video, 0, 0);
            stream.getVideoTracks()[0].stop();
            div.remove();
            return canvas.toDataURL('image/jpeg', quality);
        }
    ''')
    display(js)
    data = eval_js('takePhoto({})'.format(quality))
    binary = b64decode(data.split(',')[1])
    with open(filename, 'wb') as f:
        f.write(binary)
    return filename

btn_webcam = widgets.Button(
    description='📸 Capture Live Camera',
    button_style='primary',
    layout=widgets.Layout(width='210px', height='40px')
)

btn_upload = widgets.Button(
    description='📁 Upload Image File',
    button_style='info',
    layout=widgets.Layout(width='210px', height='40px')
)

output_area = widgets.Output()

def on_webcam_clicked(b):
    with output_area:
        clear_output()
        print("🎥 Opening webcam... Allow browser camera access and click '📸 Click to Snap Photo'.")
        captured_path = capture_webcam_colab()
        clear_output()
        predict_facial_expression(captured_path)

def on_upload_clicked(b):
    with output_area:
        clear_output()
        print("📁 Select an image file to upload...")
        uploaded = files.upload()
        if uploaded:
            clear_output()
            for fname in uploaded.keys():
                predict_facial_expression(fname)

btn_webcam.on_click(on_webcam_clicked)
btn_upload.on_click(on_upload_clicked)

display(widgets.VBox([
    widgets.HTML("<h3>Facial Expression Recognition</h3>"),
    widgets.HBox([btn_webcam, btn_upload]),
    output_area
]))